In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

# ================= CONFIGURATION =================
CSV_PATH = "handwritten_digits_8x8.csv"
LABEL_COLUMN = "label"

IMAGE_HEIGHT = 8
IMAGE_WIDTH = 8
CHANNELS = 1

BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 50
TEST_SIZE = 0.2
SEED = 42

# ================= REPRODUCIBILITY =================
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)



Device: cuda


In [ ]:

df = pd.read_csv(CSV_PATH)

label_col = (
    df.columns[0] if LABEL_COLUMN is None
    else LABEL_COLUMN
)

X = df.drop(columns=[label_col]).to_numpy(
    dtype=np.float32
)
y_raw = df[label_col].to_numpy()

expected_pixels = IMAGE_HEIGHT * IMAGE_WIDTH * CHANNELS

if X.shape[1] != expected_pixels:
    raise ValueError(
        f"Expected {expected_pixels} pixel columns, "
        f"but found {X.shape[1]}. Check image dimensions, "
        "channels, and extra columns."
    )

if not np.isfinite(X).all():
    raise ValueError("Dataset contains missing or infinite pixel values.")

# Normalize pixel values
if X.size and X.max() > 1.0:
    X = X / 255.0

# Encode class labels as integers: 0, 1, 2, ...
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
NUM_CLASSES = len(label_encoder.classes_)

print("Classes:", label_encoder.classes_)
print("Number of classes:", NUM_CLASSES)

# ================= SPLIT DATA =================


Classes: [0 1 2 3 4 5 6 7 8 9]
Number of classes: 10


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y
)

# ================= CUSTOM DATASET =================
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        features = torch.tensor(
            features, dtype=torch.float32
        )

        if CHANNELS == 1:
            self.features = features.reshape(
                -1, 1, IMAGE_HEIGHT, IMAGE_WIDTH
            )
        else:
            # Assumes flattened RGB pixels are in HWC order
            self.features = features.reshape(
                -1, IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS
            ).permute(0, 3, 1, 2).contiguous()

        self.labels = torch.tensor(
            labels, dtype=torch.long
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]




In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=(device.type == "cuda")
)

# ================= CNN MODEL =================
class MyCNN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                input_channels, 32,
                kernel_size=3, padding=1
            ),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3, padding=1
            ),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)




In [ ]:
model = MyCNN(CHANNELS, NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

# ================= TRAINING =================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {total_loss / total:.4f} "
        f"Train Accuracy: {100 * correct / total:.2f}%"
    )



Epoch [1/50] Loss: 2.0101 Train Accuracy: 31.87%
Epoch [2/50] Loss: 0.8970 Train Accuracy: 75.23%
Epoch [3/50] Loss: 0.3149 Train Accuracy: 92.07%
Epoch [4/50] Loss: 0.1615 Train Accuracy: 95.96%
Epoch [5/50] Loss: 0.1042 Train Accuracy: 97.43%
Epoch [6/50] Loss: 0.0706 Train Accuracy: 98.26%
Epoch [7/50] Loss: 0.0470 Train Accuracy: 99.10%
Epoch [8/50] Loss: 0.0414 Train Accuracy: 98.96%
Epoch [9/50] Loss: 0.0289 Train Accuracy: 99.44%
Epoch [10/50] Loss: 0.0222 Train Accuracy: 99.65%
Epoch [11/50] Loss: 0.0256 Train Accuracy: 99.30%
Epoch [12/50] Loss: 0.0238 Train Accuracy: 99.37%
Epoch [13/50] Loss: 0.0190 Train Accuracy: 99.58%
Epoch [14/50] Loss: 0.0170 Train Accuracy: 99.51%
Epoch [15/50] Loss: 0.0336 Train Accuracy: 99.16%
Epoch [16/50] Loss: 0.0257 Train Accuracy: 99.16%
Epoch [17/50] Loss: 0.0173 Train Accuracy: 99.51%
Epoch [18/50] Loss: 0.0200 Train Accuracy: 99.51%
Epoch [19/50] Loss: 0.0153 Train Accuracy: 99.72%
Epoch [20/50] Loss: 0.0080 Train Accuracy: 99.86%
Epoch [21

In [ ]:
# ================= EVALUATION =================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"\nTest Accuracy: {100 * correct / total:.2f}%")

# ================= SAVE MODEL =================
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": label_encoder.classes_.tolist(),
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "channels": CHANNELS,
        "num_classes": NUM_CLASSES
    },
    "cnn_classifier.pth"
)

print("Model saved to cnn_classifier.pth")


Test Accuracy: 99.17%
Model saved to cnn_classifier.pth


In [ ]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

1.0
